In [77]:
import json
import os
import sys
import shutil
from pathlib import Path

os.environ["MESA_DIR"] = "/Users/mchau/mesa"

print("MESA_DIR =", os.environ["MESA_DIR"])


def locate_project_root(start=Path.cwd()):
    start = Path(start).resolve()

    for candidate in (start, *start.parents):
        if (candidate / "tutorial" / "inlist_project").is_file():
            return candidate

    raise FileNotFoundError(
        "Could not locate tutorial/inlist_project"
    )


PROJECT_ROOT = locate_project_root()

PIPELINE_DIR = (
    PROJECT_ROOT
    / "scripts"
    / "mams_profiles"
    / "pipeline"
)

if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))


from mams_pipeline import (
    build_configs,
    inspect_runs,
    model_label,
    prepare_custom_run,
    run_local,
)


TEMPLATE_DIR = PROJECT_ROOT / "tutorial"

OUTPUT_ROOT = (
    PROJECT_ROOT
    / "results"
    / "mams_profiles"
).resolve()

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Template:", TEMPLATE_DIR)
print("Output:", OUTPUT_ROOT)

MESA_DIR = /Users/mchau/mesa
Project root: /Users/mchau/mesa_tutorial
Template: /Users/mchau/mesa_tutorial/tutorial
Output: /Users/mchau/mesa_tutorial/results/mams_profiles


In [78]:
template_text = (
    TEMPLATE_DIR / "inlist_project"
).read_text(encoding="utf-8")

print(template_text)

&star_job

   create_pre_main_sequence_model = .true.

   ! Specify the nuclear network
   change_initial_net = .true.
   new_net_name = 'mesa_49.net'

   ! Output

   write_profile_when_terminate = .TRUE.
   filename_for_profile_when_terminate = 'MAMS.profile'

   save_pulse_data_when_terminate = .TRUE.
   save_pulse_data_filename = 'MAMS5p1Msun.mesa'

/

&controls

   ! Starting parameters

   initial_mass = 5.1
   initial_y = 0.2703  ! Helium mass fraction
   initial_z = 0.0142  ! Metal mass fraction, hydrogen = 1 - metal - helium
   log_Rsurf_upper_limit = 1.0
    history_interval = 1

   ! Convection

   mixing_length_alpha = 2.0
   MLT_option = 'Cox'

   overshoot_f(:) = 0.0001
   overshoot_f0(:) = 0.0005

   ! Brunt-Vaisala smoothing

   num_cells_for_smooth_brunt_B = 0

   ! Control the time steps and output
   min_timestep_limit = 1.0d-20  ! minimum timestep limit
   ! varcontrol_target = 1d-3
   mesh_delta_coeff = 1.0
   ! dX_nuc_drop_min_X_limit = 1d-3

   ! Output to files 

In [79]:
STAGE1_LABEL = "RGB3Msun_Z0p0142_stage1"

stage1_inlist = template_text

# --------------------------------------------------
# Remove radius termination from the template
# --------------------------------------------------

stage1_inlist = stage1_inlist.replace(
    "log_Rsurf_upper_limit = 1",
    "! log_Rsurf_upper_limit disabled for Stage 1"
)

stage1_inlist = stage1_inlist.replace(
    "log_Rsurf_upper_limit = 1.0",
    "! log_Rsurf_upper_limit disabled for Stage 1"
)

# --------------------------------------------------
# Make sure Stage 1 creates a PMS model
# --------------------------------------------------

if "create_pre_main_sequence_model" in stage1_inlist:
    import re

    stage1_inlist = re.sub(
        r"^\s*create_pre_main_sequence_model\s*=.*$",
        "   create_pre_main_sequence_model = .true.",
        stage1_inlist,
        flags=re.MULTILINE,
    )
else:
    stage1_inlist = stage1_inlist.replace(
        "&star_job\n",
        "&star_job\n\n"
        "   create_pre_main_sequence_model = .true.\n",
        1,
    )

# --------------------------------------------------
# Remove any restart settings
# --------------------------------------------------

import re

stage1_inlist = re.sub(
    r"^\s*load_saved_model\s*=.*$\n?",
    "",
    stage1_inlist,
    flags=re.MULTILINE,
)

stage1_inlist = re.sub(
    r"^\s*saved_model_name\s*=.*$\n?",
    "",
    stage1_inlist,
    flags=re.MULTILINE,
)

# --------------------------------------------------
# Set the actual stellar parameters
# --------------------------------------------------

stage1_inlist = re.sub(
    r"^\s*initial_mass\s*=.*$",
    "   initial_mass = 3",
    stage1_inlist,
    flags=re.MULTILINE,
)

stage1_inlist = re.sub(
    r"^\s*initial_y\s*=.*$",
    "   initial_y = 0.2703",
    stage1_inlist,
    flags=re.MULTILINE,
)

stage1_inlist = re.sub(
    r"^\s*initial_z\s*=.*$",
    "   initial_z = 0.0142",
    stage1_inlist,
    flags=re.MULTILINE,
)

stage1_inlist = re.sub(
    r"^\s*Zbase\s*=.*$",
    "  Zbase = 0.0142",
    stage1_inlist,
    flags=re.MULTILINE,
)

# --------------------------------------------------
# Stage 1 termination: central H1 = 0.01
# --------------------------------------------------

stage1_inlist = re.sub(
    r"^\s*xa_central_lower_limit_species\(1\)\s*=.*$",
    "   xa_central_lower_limit_species(1) = 'h1'",
    stage1_inlist,
    flags=re.MULTILINE,
)

stage1_inlist = re.sub(
    r"^\s*xa_central_lower_limit\(1\)\s*=.*$",
    "   xa_central_lower_limit(1) = 0.01",
    stage1_inlist,
    flags=re.MULTILINE,
)

# --------------------------------------------------
# Save an actual MESA restart model
# --------------------------------------------------

if "save_model_when_terminate" in stage1_inlist:
    stage1_inlist = re.sub(
        r"^\s*save_model_when_terminate\s*=.*$",
        "   save_model_when_terminate = .true.",
        stage1_inlist,
        flags=re.MULTILINE,
    )
else:
    stage1_inlist = stage1_inlist.replace(
        "&star_job\n",
        "&star_job\n\n"
        "   save_model_when_terminate = .true.\n"
        "   save_model_filename = 'RGB_restart.mod'\n",
        1,
    )

if "save_model_filename" in stage1_inlist:
    stage1_inlist = re.sub(
        r"^\s*save_model_filename\s*=.*$",
        "   save_model_filename = 'RGB_restart.mod'",
        stage1_inlist,
        flags=re.MULTILINE,
    )
else:
    stage1_inlist = stage1_inlist.replace(
        "   save_model_when_terminate = .true.\n",
        "   save_model_when_terminate = .true.\n"
        "   save_model_filename = 'RGB_restart.mod'\n",
        1,
    )

print(stage1_inlist)

&star_job

   save_model_when_terminate = .true.
   save_model_filename = 'RGB_restart.mod'
   create_pre_main_sequence_model = .true.

   ! Specify the nuclear network
   change_initial_net = .true.
   new_net_name = 'mesa_49.net'

   ! Output

   write_profile_when_terminate = .TRUE.
   filename_for_profile_when_terminate = 'MAMS.profile'

   save_pulse_data_when_terminate = .TRUE.
   save_pulse_data_filename = 'MAMS5p1Msun.mesa'

/

&controls

   ! Starting parameters
   initial_mass = 3
   initial_y = 0.2703
   initial_z = 0.0142
   ! log_Rsurf_upper_limit disabled for Stage 1.0
    history_interval = 1

   ! Convection

   mixing_length_alpha = 2.0
   MLT_option = 'Cox'

   overshoot_f(:) = 0.0001
   overshoot_f0(:) = 0.0005

   ! Brunt-Vaisala smoothing

   num_cells_for_smooth_brunt_B = 0

   ! Control the time steps and output
   min_timestep_limit = 1.0d-20  ! minimum timestep limit
   ! varcontrol_target = 1d-3
   mesh_delta_coeff = 1.0
   ! dX_nuc_drop_min_X_limit = 1d-3

  

In [80]:
print("Running Stage 1...")
run_local(
    type("Run", (), {"run_dir": stage1_dir, "label": STAGE1_LABEL})()
)

Running Stage 1...


CompletedProcess(args=['./run.sh'], returncode=0, stdout='', stderr='')

In [81]:
import re

STAGE2_LABEL = "RGB3Msun_Z0p0142"

stage2_inlist = template_text

# --------------------------------------------------
# Remove PMS creation
# --------------------------------------------------

stage2_inlist = re.sub(
    r"^\s*create_pre_main_sequence_model\s*=.*$\n?",
    "",
    stage2_inlist,
    flags=re.MULTILINE,
)

# --------------------------------------------------
# Remove any existing restart settings
# --------------------------------------------------

stage2_inlist = re.sub(
    r"^\s*load_saved_model\s*=.*$\n?",
    "",
    stage2_inlist,
    flags=re.MULTILINE,
)

stage2_inlist = re.sub(
    r"^\s*load_model_filename\s*=.*$\n?",
    "",
    stage2_inlist,
    flags=re.MULTILINE,
)

stage2_inlist = re.sub(
    r"^\s*saved_model_name\s*=.*$\n?",
    "",
    stage2_inlist,
    flags=re.MULTILINE,
)

# --------------------------------------------------
# Add the CORRECT MESA restart settings
# --------------------------------------------------

stage2_inlist = stage2_inlist.replace(
    "&star_job\n",
    "&star_job\n\n"
    "   load_saved_model = .true.\n"
    "   load_model_filename = 'RGB_restart.mod'\n",
    1,
)

# --------------------------------------------------
# Remove initial stellar parameters
# --------------------------------------------------

stage2_inlist = re.sub(
    r"^\s*initial_mass\s*=.*$\n?",
    "",
    stage2_inlist,
    flags=re.MULTILINE,
)

stage2_inlist = re.sub(
    r"^\s*initial_y\s*=.*$\n?",
    "",
    stage2_inlist,
    flags=re.MULTILINE,
)

stage2_inlist = re.sub(
    r"^\s*initial_z\s*=.*$\n?",
    "",
    stage2_inlist,
    flags=re.MULTILINE,
)

# --------------------------------------------------
# Set 10 R_sun termination
# --------------------------------------------------

stage2_inlist = re.sub(
    r"^\s*log_Rsurf_upper_limit\s*=.*$",
    "   log_Rsurf_upper_limit = 1.0",
    stage2_inlist,
    flags=re.MULTILINE,
)

if "log_Rsurf_upper_limit" not in stage2_inlist:
    stage2_inlist = stage2_inlist.replace(
        "&controls\n",
        "&controls\n\n"
        "   log_Rsurf_upper_limit = 1.0\n",
        1,
    )

# --------------------------------------------------
# Correct metallicity
# --------------------------------------------------

stage2_inlist = re.sub(
    r"^\s*Zbase\s*=.*$",
    "   Zbase = 0.0142",
    stage2_inlist,
    flags=re.MULTILINE,
)

print(stage2_inlist)

# Remove the Stage 2 central-H stopping condition
stage2_inlist = stage2_inlist.replace(
    """
   ! When to stop

   xa_central_lower_limit_species(1) = 'h1'
   xa_central_lower_limit(1) = 0.35
""",
    """
   ! When to stop
   ! No central-H limit in Stage 2;
   ! Stage 2 stops when the star reaches 10 R_sun.
"""
)

print(stage2_inlist)

&star_job

   load_saved_model = .true.
   load_model_filename = 'RGB_restart.mod'

   ! Specify the nuclear network
   change_initial_net = .true.
   new_net_name = 'mesa_49.net'

   ! Output

   write_profile_when_terminate = .TRUE.
   filename_for_profile_when_terminate = 'MAMS.profile'

   save_pulse_data_when_terminate = .TRUE.
   save_pulse_data_filename = 'MAMS5p1Msun.mesa'

/

&controls

   ! Starting parameters
   log_Rsurf_upper_limit = 1.0
    history_interval = 1

   ! Convection

   mixing_length_alpha = 2.0
   MLT_option = 'Cox'

   overshoot_f(:) = 0.0001
   overshoot_f0(:) = 0.0005

   ! Brunt-Vaisala smoothing

   num_cells_for_smooth_brunt_B = 0

   ! Control the time steps and output
   min_timestep_limit = 1.0d-20  ! minimum timestep limit
   ! varcontrol_target = 1d-3
   mesh_delta_coeff = 1.0
   ! dX_nuc_drop_min_X_limit = 1d-3

   ! Output to files and terminal

   ! photostep = 0
   profile_interval = 0
   history_interval = 0
   terminal_interval = 10
   write_

In [82]:
assert "load_saved_model = .true." in stage2_inlist
assert "load_model_filename = 'RGB_restart.mod'" in stage2_inlist

assert "saved_model_name" not in stage2_inlist
assert "create_pre_main_sequence_model" not in stage2_inlist

assert "initial_mass" not in stage2_inlist
assert "initial_y" not in stage2_inlist
assert "initial_z" not in stage2_inlist

assert "log_Rsurf_upper_limit = 1.0" in stage2_inlist
assert "Zbase = 0.0142" in stage2_inlist

print("Stage 2 validation passed.")

Stage 2 validation passed.


In [83]:
stage2_dir = prepare_custom_run(
    STAGE2_LABEL,
    stage2_inlist,
    TEMPLATE_DIR,
    OUTPUT_ROOT,
    overwrite=True,
)

print(stage2_dir)

/Users/mchau/mesa_tutorial/results/mams_profiles/RGB3Msun_Z0p0142


In [84]:
restart_model = stage1_dir / "RGB_restart.mod"

if not restart_model.is_file():
    raise FileNotFoundError(
        f"Could not find Stage 1 restart model: {restart_model}"
    )

shutil.copy2(
    restart_model,
    stage2_dir / "RGB_restart.mod"
)

print("Restart model copied successfully.")

Restart model copied successfully.


In [85]:
print(
    (stage2_dir / "inlist_project").read_text(
        encoding="utf-8"
    )
)

print("\nRestart exists:", (stage2_dir / "RGB_restart.mod").is_file())

&star_job

   load_saved_model = .true.
   load_model_filename = 'RGB_restart.mod'

   ! Specify the nuclear network
   change_initial_net = .true.
   new_net_name = 'mesa_49.net'

   ! Output

   write_profile_when_terminate = .TRUE.
   filename_for_profile_when_terminate = 'MAMS.profile'

   save_pulse_data_when_terminate = .TRUE.
   save_pulse_data_filename = 'MAMS5p1Msun.mesa'

/

&controls

   ! Starting parameters
   log_Rsurf_upper_limit = 1.0
    history_interval = 1

   ! Convection

   mixing_length_alpha = 2.0
   MLT_option = 'Cox'

   overshoot_f(:) = 0.0001
   overshoot_f0(:) = 0.0005

   ! Brunt-Vaisala smoothing

   num_cells_for_smooth_brunt_B = 0

   ! Control the time steps and output
   min_timestep_limit = 1.0d-20  ! minimum timestep limit
   ! varcontrol_target = 1d-3
   mesh_delta_coeff = 1.0
   ! dX_nuc_drop_min_X_limit = 1d-3

   ! Output to files and terminal

   ! photostep = 0
   profile_interval = 0
   history_interval = 0
   terminal_interval = 10
   write_

In [86]:
# --------------------------------------------------
# Run Stage 2
# --------------------------------------------------

print(f"Running {STAGE2_LABEL} ...", flush=True)

result = run_local(
    type(
        "PreparedRun",
        (),
        {
            "run_dir": stage2_dir,
            "label": STAGE2_LABEL,
        },
    )(),
    check=False,
)

print("\nReturn code:", result.returncode)

if result.stdout:
    print("\n========== STDOUT ==========")
    print(result.stdout)

if result.stderr:
    print("\n========== STDERR ==========")
    print(result.stderr)

Running RGB3Msun_Z0p0142 ...

Return code: 0


In [67]:
import subprocess
from pathlib import Path

stage2_dir = Path(stage2_dir).resolve()

result = subprocess.run(
    ["./run.sh"],
    cwd=stage2_dir,
    text=True,
    capture_output=True
)

print("Return code:", result.returncode)
print("\nSTDOUT:")
print(result.stdout[-10000:])

print("\nSTDERR:")
print(result.stderr[-10000:])

print("\nFiles after run:")
for p in sorted(stage2_dir.iterdir()):
    print(" ", p.name)

Return code: 0

STDOUT:


STDERR:


Files after run:
  .ipynb_checkpoints
  .mesa_temp_cache
  LOGS
  MAMS.profile
  MAMS5p1Msun.mesa
  RGB_restart.mod
  inlist
  inlist_pgstar
  inlist_project
  mesa.err
  mesa.out
  photos
  run.sh
  star


In [71]:
from pathlib import Path

mesa_file = Path(stage2_dir) / "MAMS5p1Msun.mesa"

with open(mesa_file, "r") as f:
    for i in range(30):
        line = f.readline()
        if not line:
            break
        print(f"{i+1:02d}: {line.rstrip()}")

01:   1273     5.9652296120941509E+33     7.1132096299211426E+11     1.0887831988425086E+36    101
02:      1     0.0000000000000000E+00     0.0000000000000000E+00     0.0000000000000000E+00     1.3518316743354486E+20     7.9201525133835554E+07     2.2763490439884336E+04     2.1271383459132850E-01     0.0000000000000000E+00     1.6502580723347582E+00     3.9020116629849844E-01     6.4187103450440508E-01     2.5860335723830685E-01    -3.0683426671042258E-01     4.3142206972331655E-02     2.3330985056487299E-02     1.7480275521433683E-01     2.3093450495247813E-02     0.0000000000000000E+00
03:      2     1.6971723870448496E+07     4.6609619618679384E+26     2.8305669963493868E+28     1.3516307039129892E+20     7.9199053616148517E+07     2.2761086562570381E+04     2.1271224694923518E-01     2.2378078955429988E-07     1.6502588135024936E+00     3.9020137928975507E-01     6.4188534091721305E-01     2.5860837597536618E-01    -3.0684531439734830E-01     4.3146041556098851E-02     2.338177006